# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Treating this as a Binary classification problem will be a good strategy for better ranking.

The target is "is_declining_label", 1 means the content item is declining and 0 means it is not. In this way i can use the model's probablity of decline to rank the pages.
I will begin with logistic regression because it is appropriate for a yes/no outcome and its coefficients are easier to interpret. Then I will compare it with a random forest because random forests can capture nonlinear relationships between traffic, content age, search position, and engagement. The comparison will use the same data split and the same metrics as my baseline. I will focus especially on precision@20 and precision@50 because the practical goal is to identify the best pages to review first.

In [2]:

import pandas as pd
from pathlib import Path
RANDOM_STATE = 42

repo_root =Path(r"D:\git\Assignment-1")
feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"

if not feature_path.exists():
    raise FileNotFoundError(
        "Processed feature file is missing. Run this first in the terminal:\n"
        "py -m pip install -r requirements.txt\n"
        "py scripts/run_all.py\n"
        f"Expected path: {feature_path}"
    )

df =pd.read_csv(feature_path)
y =df["is_declining_label"].astype(int)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

X_numeric = df[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)
X_categorical = pd.get_dummies(
    df[categorical_features].fillna("unknown").astype(str),
    dtype=float,
)
X = pd.concat(
    [X_numeric.reset_index(drop=True),
     X_categorical.reset_index(drop=True)],
    axis=1,
)

print(f"Rows: {len(df):,}")
print(f"Features: {X.shape[1]:,}")
print(f"Declining base rate: {y.mean():.1%}")


Rows: 30,000
Features: 52
Declining base rate: 54.2%


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a grouped client holdout split for this model.

The unit of prediction is a content item, but pages from the same client are related and can share traffic patterns, search behavior, and content strategy. If I randomly split rows, the model could see similar pages from the same client in both train and test, which would make the score look stronger than it really is. To keep the evaluation honest, I will hold out entire clients for the test set and train on the remaining clients.

This is a realistic evaluation for a refresh lane because the model is meant to help prioritize review for new or unseen client portfolios rather than memorize patterns that already appear in the same client group.

I will keep the same client-based split when comparing the model against the baseline so the comparison is fair.

In [3]:
import numpy as np

df = df[["client_id", "is_declining_label"]].copy()
df["is_declining_label"] = df["is_declining_label"].astype(int)
RANDOM_STATE = 42
clients = df["client_id"].dropna().unique()
rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(clients)
test_clients_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_clients_count])

df["split"] = "train"
df.loc[df["client_id"].isin(test_clients), "split"] = "test"

train_df = df[df["split"] == "train"]
test_df = df[df["split"] == "test"]

print(f"Total rows: {len(df):,}")
print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Train clients: {train_df['client_id'].nunique():,}")
print(f"Test clients: {test_df['client_id'].nunique():,}")
print(f"Positive rate in train: {train_df['is_declining_label'].mean():.2%}")
print(f"Positive rate in test: {test_df['is_declining_label'].mean():.2%}")
print("Split type: client holdout")

Total rows: 30,000
Train rows: 27,675
Test rows: 2,325
Train clients: 26
Test clients: 6
Positive rate in train: 55.48%
Positive rate in test: 39.10%
Split type: client holdout


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare the learned model against the Week-4 baseline using the same client-holdout test set and the same ranking metric. The baseline score is the hand-written refresh rule, where as the model score is the predicted probability of a page being in the declining category. I will report precision@20 and precision@50 because the actual question is which pages should be reviewed first.
The comparison is fair because both methods are evaluated on the same hold-out clients and the ranking metric is the same. If the learned model improves precision@20 or precision@50 over the baseline, that is evidence that it is more practical for refresh prioritization.

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE =42
repo_root = Path(r"D:\git\Assignment-1")
df= pd.read_csv(repo_root / "data" / "processed" / "refresh_feature_vector.csv")
df= df[
    [
        "client_id",
        "content_id",
        "is_declining_label",
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "trend_direction",
        "search_volume",
        "competition",
        "cpc",
        "word_count",
        "char_count",
        "log_impressions_90d",
        "log_clicks_90d",
        "log_sessions_90d",
        "log_ai_sessions_90d",
        "days_with_impressions",
        "days_with_sessions",
        "content_age_days",
        "ctr",
        "engagement_rate",
        "scroll_rate",
        "ai_traffic_pct",
        "competition_level",
        "content_type",
        "main_intent",
        "age_tier",
        "freshness_tier",
        "word_count_tier",
        "impression_tier",
        "position_tier",
    ]
].copy()
df["signal_stale"] =(df["days_since_last_update"]>= 180).astype(int)
df["signal_previously_strong"] = (df["impressions_90d"]>= 500).astype(int)
df["signal_declining"]= (df["trend_direction"]== "down").astype(int)

df["baseline_score"] = (
    df["signal_stale"]
    * df["signal_previously_strong"]
    * df["signal_declining"]
    * df["impressions_90d"]
)
clients = df["client_id"].dropna().unique()
rng =np.random.default_rng(RANDOM_STATE)
shuffled_clients= rng.permutation(clients)
test_clients_count =max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients =set(shuffled_clients[:test_clients_count])

df["split"]="train"
df.loc[df["client_id"].isin(test_clients), "split"] = "test"

train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

numeric_features =[
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features=[
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

X_numeric_train= train_df[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)
X_numeric_test=test_df[numeric_features].apply(pd.to_numeric, errors="coerce").fillna(0)

X_cat_train =pd.get_dummies(train_df[categorical_features].fillna("unknown").astype(str), dtype=float)
X_cat_test =pd.get_dummies(test_df[categorical_features].fillna("unknown").astype(str), dtype=float)

X_cat_train, X_cat_test = X_cat_train.align(X_cat_test, join="outer", axis=1, fill_value=0)

X_train = pd.concat([X_numeric_train.reset_index(drop=True), X_cat_train.reset_index(drop=True)], axis=1)
X_test = pd.concat([X_numeric_test.reset_index(drop=True), X_cat_test.reset_index(drop=True)], axis=1)

y_train = train_df["is_declining_label"].astype(int)
y_test = test_df["is_declining_label"].astype(int)

baseline_scores = test_df["baseline_score"].to_numpy()

models={
    "logistic_regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))
        ]
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

def precision_at_k(y_true, scores, k):
    if len(scores)==0:
        return 0.0
    frame =pd.DataFrame({"y": y_true, "score": scores})
    top =frame.sort_values("score", ascending=False).head(k)
    return float(top["y"].mean()) if len(top) else 0.0

rows = []

rows.append({
    "method":"baseline",
    "precision_at_20": precision_at_k(y_test, baseline_scores, 20),
    "precision_at_50": precision_at_k(y_test, baseline_scores, 50),
})

for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    rows.append({
        "method":name,
        "precision_at_20": precision_at_k(y_test, prob, 20),
        "precision_at_50": precision_at_k(y_test, prob, 50),
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

             method  precision_at_20  precision_at_50
           baseline             0.30             0.40
logistic_regression             0.30             0.36
      random_forest             0.65             0.70


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model can make two important types of errors. A false positive is a page the model ranks as posssibly declining when the observed label is 0. A false negative is a declining page that the fails to rank higher.
I will inspect examples of both error types and examinanalyze the model's most important features. This helps in finding out whether the model is using reasonable refresh signals such as traffic, freshness, position, and engagement, or if it is possibly relying on suspicious or weaker information.
These results should be treated as directional decision-support evidence. A high model score does not prove that refreshing a page will improve its future performance.

In [5]:

selected_model = model
test_probabilities = prob

error_frame = test_df[
    [
        "content_id",
        "client_id",
        "is_declining_label",
        "impressions_90d",
        "days_since_last_update",
        "avg_position",
        "trend_direction",
    ]
].copy()

error_frame["predicted_probability"] = test_probabilities
error_frame["predicted_label"] = (
    error_frame["predicted_probability"] >= 0.5
).astype(int)

error_frame["error_type"] = np.select(
    [
        (error_frame["predicted_label"] == 1)
        & (error_frame["is_declining_label"] == 0),
        (error_frame["predicted_label"] == 0)
        & (error_frame["is_declining_label"] == 1),
    ],
    [
        "false_positive",
        "false_negative",
    ],
    default="correct",
)

false_positives = (
    error_frame[error_frame["error_type"] == "false_positive"]
    .sort_values("predicted_probability", ascending=False)
    .head(3)
)

print("TOP FALSE POSITIVES")
print(false_positives.to_string(index=False))

false_negatives = (
    error_frame[error_frame["error_type"] == "false_negative"]
    .sort_values("predicted_probability", ascending=True)
    .head(3)
)

print("\nTOP FALSE NEGATIVES")
print(false_negatives.to_string(index=False))

print("\nERROR COUNTS")
print(error_frame["error_type"].value_counts())

if hasattr(selected_model, "feature_importances_"):
    importance_values = selected_model.feature_importances_
else:
    importance_values = np.zeros(X_train.shape[1])

feature_importance = (
    pd.DataFrame(
        {
            "feature": X_train.columns,
            "importance": importance_values,
        }
    )
    .sort_values("importance", ascending=False)
    .head(10)
)

print("\nTOP MODEL FEATURES")
print(feature_importance.to_string(index=False))

TOP FALSE POSITIVES
          content_id         client_id  is_declining_label  impressions_90d  days_since_last_update  avg_position trend_direction  predicted_probability  predicted_label     error_type
content_331182ca4cae client_f74efabef1                   0             3026                      20          35.9              up               0.756866                1 false_positive
content_d2dffcc697a4 client_f74efabef1                   0             5091                      20          14.1          stable               0.745062                1 false_positive
content_db1cd41b4b4f client_f74efabef1                   0             1482                     105          12.9              up               0.743844                1 false_positive

TOP FALSE NEGATIVES
          content_id         client_id  is_declining_label  impressions_90d  days_since_last_update  avg_position trend_direction  predicted_probability  predicted_label     error_type
content_f1ef151d5e36 client_d4735e

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.